ModuleNotFoundError: No module named 'nnfs'

In [11]:
import numpy as np

# ============================================================
# LAYER
# ============================================================

class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        # small random weights — shape(n_inputs, n_neurons)
        # pre-aligned so forward pass needs no transpose
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        
        # zero biases — shape(1, n_neurons) for broadcasting
        self.biases = np.zeros((1, n_neurons))

    def forward(self, inputs):
        self.inputs = inputs  # save for backprop later
        # core operation: (batch, inputs) x (inputs, neurons) = (batch, neurons)
        self.output = np.dot(inputs, self.weights) + self.biases


# ============================================================
# ACTIVATIONS
# ============================================================

class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs  # save for backprop — we need to know which were negative
        # element-wise max: negatives → 0, positives → unchanged
        self.output = np.maximum(0, inputs)


class Activation_Softmax:
    def forward(self, inputs):
        self.inputs = inputs
        
        # subtract max per sample BEFORE exp — prevents overflow (e^1000 = inf)
        # axis=1 → across neurons per sample
        # keepdims=True → keeps shape (batch,1) for broadcasting
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        
        # normalize: divide each by sum of its row → all rows sum to 1
        # now these are proper probabilities
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        
        self.output = probabilities


# ============================================================
# LOSS — BASE CLASS
# ============================================================

class Loss:
    def calculate(self, output, y):
        # output = network predictions (softmax probabilities)
        # y = true labels (ground truth)
        
        # call the child class forward() to get per-sample losses
        sample_losses = self.forward(output, y)
        
        # average across all samples → single loss number
        data_loss = np.mean(sample_losses)
        
        return data_loss


# ============================================================
# CATEGORICAL CROSS-ENTROPY LOSS
# ============================================================

class Loss_CategoricalCrossentropy(Loss):
    
    def forward(self, y_pred, y_true):
        # y_pred shape: (batch_size, n_classes) — softmax probabilities
        # y_true shape: (batch_size,) if sparse OR (batch_size, n_classes) if one-hot
        
        samples = len(y_pred)  # number of samples in batch
        
        # clip predictions to avoid log(0) = -infinity which breaks everything
        # clip both sides so we don't accidentally push mean toward any value
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)
        
        # CASE 1: sparse labels — y_true = [0, 2, 1] (class indices)
        if len(y_true.shape) == 1:
            # pick the predicted probability for the correct class only
            # range(samples) = [0,1,2,...] → row index
            # y_true = [0,2,1] → column index (correct class)
            correct_confidences = y_pred_clipped[range(samples), y_true]
        
        # CASE 2: one-hot labels — y_true = [[1,0,0],[0,0,1],[0,1,0]]
        elif len(y_true.shape) == 2:
            # multiply prediction by one-hot mask → zeros out wrong classes
            # sum across axis=1 → gives correct class probability per sample
            correct_confidences = np.sum(y_pred_clipped * y_true, axis=1)
        
        # apply negative log — higher confidence = lower loss
        negative_log_likelihoods = -np.log(correct_confidences)
        
        return negative_log_likelihoods  # shape: (batch_size,)


# ============================================================
# FULL FORWARD PASS — EVERYTHING CONNECTED
# ============================================================

# --- create spiral dataset ---
# using nnfs for reproducible spiral data
import nnfs
from nnfs.datasets import spiral_data
nnfs.init()  # sets random seed + float32 default

X, y = spiral_data(samples=100, classes=3)
# X shape: (300, 2) — 300 samples, 2 features (x,y coordinates)
# y shape: (300,)  — class labels 0,1,2

# --- build network ---
dense1   = Layer_Dense(2, 3)        # 2 inputs → 3 neurons
relu1    = Activation_ReLU()        # hidden activation
dense2   = Layer_Dense(3, 3)        # 3 inputs → 3 outputs (3 classes)
softmax  = Activation_Softmax()     # output activation → probabilities

# --- forward pass ---
dense1.forward(X)                   # (300,2) → (300,3)
relu1.forward(dense1.output)        # (300,3) → (300,3) negatives zeroed

dense2.forward(relu1.output)        # (300,3) → (300,3)
softmax.forward(dense2.output)      # (300,3) → (300,3) probabilities

# --- calculate loss ---
loss_function = Loss_CategoricalCrossentropy()
loss = loss_function.calculate(softmax.output, y)

print(f"Predictions (first 5 samples):\n{softmax.output[:5]}")
print(f"\nTrue labels (first 5): {y[:5]}")
print(f"\nLoss: {loss:.4f}")


# ============================================================
# ACCURACY — BONUS: HOW MANY DID WE GET RIGHT
# ============================================================

# get predicted class = index of highest probability per sample
predictions = np.argmax(softmax.output, axis=1)  
# axis=1 → find max across classes for each sample

# if y is one-hot, convert to sparse first
if len(y.shape) == 2:
    y = np.argmax(y, axis=1)

# compare predictions to true labels → True/False array → mean = accuracy
accuracy = np.mean(predictions == y)

print(f"Accuracy: {accuracy:.4f}")

ModuleNotFoundError: No module named 'nnfs'

In [2]:
! pip install nnfs

In [14]:
# ============================================================
# COMPLETE NEURAL NETWORK FROM SCRATCH (FORWARD PASS ONLY)
# ============================================================

import numpy as np
import nnfs
from nnfs.datasets import spiral_data

# initialize nnfs settings
# sets random seed + float32 defaults
nnfs.init()


# ============================================================
# DENSE LAYER
# ============================================================

class Layer_Dense:

    def __init__(self, n_inputs, n_neurons):

        # initialize weights with small random values
        # shape: (inputs, neurons)
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)

        # initialize biases as zeros
        # shape: (1, neurons)
        self.biases = np.zeros((1, n_neurons))

    def forward(self, inputs):

        # save inputs for future backpropagation
        self.inputs = inputs

        # linear transformation
        # output = XW + b
        self.output = np.dot(inputs, self.weights) + self.biases


# ============================================================
# RELU ACTIVATION
# ============================================================

class Activation_ReLU:

    def forward(self, inputs):

        # save inputs
        self.inputs = inputs

        # apply ReLU activation
        # negative values become 0
        self.output = np.maximum(0, inputs)


# ============================================================
# SOFTMAX ACTIVATION
# ============================================================

class Activation_Softmax:

    def forward(self, inputs):

        # save inputs
        self.inputs = inputs

        # subtract max value for numerical stability
        exp_values = np.exp(
            inputs - np.max(inputs, axis=1, keepdims=True)
        )

        # normalize probabilities
        probabilities = exp_values / np.sum(
            exp_values,
            axis=1,
            keepdims=True
        )

        self.output = probabilities


# ============================================================
# BASE LOSS CLASS
# ============================================================

class Loss:

    def calculate(self, output, y):

        # calculate sample losses
        sample_losses = self.forward(output, y)

        # calculate mean loss
        data_loss = np.mean(sample_losses)

        return data_loss


# ============================================================
# CATEGORICAL CROSS-ENTROPY LOSS
# ============================================================

class Loss_CategoricalCrossentropy(Loss):

    def forward(self, y_pred, y_true):

        # number of samples
        samples = len(y_pred)

        # clip values to avoid log(0)
        y_pred_clipped = np.clip(
            y_pred,
            1e-7,
            1 - 1e-7
        )

        # ----------------------------------------------------
        # CASE 1: SPARSE LABELS
        # y_true = [0,1,2]
        # ----------------------------------------------------

        if len(y_true.shape) == 1:

            correct_confidences = y_pred_clipped[
                range(samples),
                y_true
            ]

        # ----------------------------------------------------
        # CASE 2: ONE-HOT LABELS
        # y_true = [[1,0,0],[0,1,0],...]
        # ----------------------------------------------------

        elif len(y_true.shape) == 2:

            correct_confidences = np.sum(
                y_pred_clipped * y_true,
                axis=1
            )

        # calculate negative log likelihood
        negative_log_likelihoods = -np.log(correct_confidences)

        return negative_log_likelihoods


# ============================================================
# CREATE DATASET
# ============================================================

# generate spiral dataset
X, y = spiral_data(samples=100, classes=3)

# X shape = (300,2)
# y shape = (300,)


# ============================================================
# BUILD NETWORK
# ============================================================

# first dense layer
dense1 = Layer_Dense(2, 3)

# activation function
activation1 = Activation_ReLU()

# second dense layer
dense2 = Layer_Dense(3, 3)

# output activation
activation2 = Activation_Softmax()

# loss function
loss_function = Loss_CategoricalCrossentropy()


# ============================================================
# FORWARD PASS
# ============================================================

# layer 1
dense1.forward(X)

# activation 1
activation1.forward(dense1.output)

# layer 2
dense2.forward(activation1.output)

# output activation
activation2.forward(dense2.output)


# ============================================================
# LOSS CALCULATION
# ============================================================

loss = loss_function.calculate(
    activation2.output,
    y
)


# ============================================================
# PREDICTIONS + ACCURACY
# ============================================================

# predicted class index
predictions = np.argmax(
    activation2.output,
    axis=1
)

# if labels are one-hot encoded
if len(y.shape) == 2:
    y = np.argmax(y, axis=1)

# calculate accuracy
accuracy = np.mean(predictions == y)


# ============================================================
# OUTPUT RESULTS
# ============================================================

print("First 5 softmax probabilities:\n")
print(activation2.output[:5])

print("\nFirst 5 predictions:")
print(predictions[:5])

print("\nFirst 5 true labels:")
print(y[:5])

print(f"\nLoss: {loss:.4f}")

print(f"Accuracy: {accuracy:.4f}")

ModuleNotFoundError: No module named 'nnfs'

In [16]:
!pip install nnfs
import sys
!{sys.executable} -m pip install nnfs

'c:\Users\DELL\OneDrive\Desktop\Neural' is not recognized as an internal or external command,
operable program or batch file.


In [15]:
import nnfs

print("NNFS installed successfully!")
print(nnfs.__file__)

ModuleNotFoundError: No module named 'nnfs'